# FlyPose-SAR — Fase 3: Preparazione Dataset + Training Animali

## Architettura: Dual-Stream
- **Stream 1 (Persone)**: Fase 2b — congelato
- **Stream 2 (Animali)**: questo notebook

## Approccio keypoints
Usiamo i keypoints GT di AP-10K (17pt COCO animali) mappati sul
schema FlyPose 20pt quadrupede — senza teacher esterno.
I 3 keypoints non mappabili (collo, garrese, zampa_post_dx)
restano a visibilita'=0 per design — non vengono stimati
geometricamente perche' non validi per animali in movimento.

## Specie target (fauna calabrese SAR)
cane, gatto, pecora, cinghiale (pig), volpe (fox), lupo (wolf),
capriolo (deer), cavallo (horse)

## Nota su epoche e convergenza
Test di debug con 2 epoche ha mostrato pose_loss molto piu' alta
di box_loss (9.8 vs 1.4) — la pose estimation su 20kpt/8 specie
eterogenee e' un task piu' difficile della detection e richiede
molte piu' epoche per convergere. Si usa epochs=100.

## Cella 1 — Installazione

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## Cella 2 — Configurazione path AP-10K

In [ ]:
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

AP10K_BASE = Path('/kaggle/input/datasets/imllti/ap-10k/ap-10k')
AP10K_IMGS = AP10K_BASE / 'data'

# Usa TUTTI gli split disponibili (train+val) per massimizzare i dati
AP10K_ANN_FILES = [
    AP10K_BASE / 'annotations' / 'ap10k-train-split1.json',
    AP10K_BASE / 'annotations' / 'ap10k-train-split2.json',
    AP10K_BASE / 'annotations' / 'ap10k-train-split3.json',
    AP10K_BASE / 'annotations' / 'ap10k-val-split1.json',
    AP10K_BASE / 'annotations' / 'ap10k-val-split2.json',
    AP10K_BASE / 'annotations' / 'ap10k-val-split3.json',
]

OUT_DIR = KAGGLE_WORKING / 'dataset_animal_fase3'

print(f'Immagini    : {AP10K_IMGS} ({len(list(AP10K_IMGS.glob("*.jpg")))} jpg)')
for f in AP10K_ANN_FILES:
    print(f'  {f.name}: esiste={f.exists()}')
print(f'Output      : {OUT_DIR}')

## Cella 3 — Configurazione specie e mapping keypoints

In [ ]:
import numpy as np

# ── Specie target (fauna calabrese) ─────────────────────────────────────────
# Nomi categoria AP-10K verificati nel dataset reale
SPECIES_TARGET = {
    'dog', 'cat', 'sheep', 'pig', 'fox', 'wolf', 'deer', 'horse',
}

N_KPT = 20  # schema FlyPose quadrupede

# ── Mapping AP-10K 17kpt -> FlyPose 20kpt ───────────────────────────────────
#
# AP-10K keypoints (17):
#   0:nose  1:l_eye  2:r_eye  3:l_ear  4:r_ear
#   5:l_shoulder  6:r_shoulder  7:l_elbow  8:r_elbow
#   9:l_front_paw  10:r_front_paw  11:root_of_tail
#   12:l_hip  13:r_hip  14:l_knee  15:r_knee  16:l_back_paw
#
# FlyPose 20kpt:
#   0:naso  1:occhio_sx  2:occhio_dx  3:orecchio_sx  4:orecchio_dx
#   5:collo  6:spalla_sx  7:spalla_dx  8:garrese  9:groppa
#   10:gomito_sx  11:gomito_dx  12:zampa_ant_sx  13:zampa_ant_dx
#   14:anca_sx  15:anca_dx  16:ginocchio_sx  17:ginocchio_dx
#   18:zampa_post_sx  19:zampa_post_dx
#
# Nota: keypoints 5(collo), 8(garrese), 19(zampa_post_dx) non hanno
# corrispondente in AP-10K e restano a visibilita'=0 per design.
# Non vengono stimati geometricamente (es. speculare) perche' non
# validi per animali in movimento/posizioni asimmetriche (scenari SAR).

AP10K_TO_FP20 = {
    0:  0,   # nose         -> naso
    1:  1,   # l_eye        -> occhio_sx
    2:  2,   # r_eye        -> occhio_dx
    3:  3,   # l_ear        -> orecchio_sx
    4:  4,   # r_ear        -> orecchio_dx
    5:  6,   # l_shoulder   -> spalla_sx
    6:  7,   # r_shoulder   -> spalla_dx
    7:  10,  # l_elbow      -> gomito_sx
    8:  11,  # r_elbow      -> gomito_dx
    9:  12,  # l_front_paw  -> zampa_ant_sx
    10: 13,  # r_front_paw  -> zampa_ant_dx
    11: 9,   # root_of_tail -> groppa
    12: 14,  # l_hip        -> anca_sx
    13: 15,  # r_hip        -> anca_dx
    14: 16,  # l_knee       -> ginocchio_sx
    15: 17,  # r_knee       -> ginocchio_dx
    16: 18,  # l_back_paw   -> zampa_post_sx
}

def map_ap10k_to_fp20(kpts_17: np.ndarray) -> np.ndarray:
    """kpts_17: (17,3) [x,y,v] -> (20,3) schema FlyPose"""
    out = np.zeros((N_KPT, 3), dtype=np.float32)
    for src, dst in AP10K_TO_FP20.items():
        if src < len(kpts_17):
            out[dst] = kpts_17[src]
    return out

print(f'Schema FlyPose 20pt configurato')
print(f'Specie target: {sorted(SPECIES_TARGET)}')

## Cella 4 — Caricamento e parsing annotazioni AP-10K (tutti gli split)

In [ ]:
import json
from collections import Counter

records = []
skipped = 0
species_counter = Counter()
seen_ann_ids = set()  # evita duplicati tra split

for ann_file in AP10K_ANN_FILES:
    if not ann_file.exists():
        print(f'[SKIP] {ann_file.name} non trovato')
        continue
    print(f'[*] Caricamento {ann_file.name}...')
    with open(ann_file) as f:
        data = json.load(f)

    id2file = {img['id']: img['file_name'] for img in data['images']}
    id2cat  = {cat['id']: cat['name'].lower() for cat in data.get('categories', [])}

    for ann in data['annotations']:
        ann_id = ann.get('id', -1)
        if ann_id in seen_ann_ids:
            continue
        seen_ann_ids.add(ann_id)

        cat_id  = ann.get('category_id', 0)
        species = id2cat.get(cat_id, 'unknown')
        species_counter[species] += 1

        if species not in SPECIES_TARGET:
            skipped += 1
            continue

        bbox = ann.get('bbox', [])
        if len(bbox) < 4 or bbox[2] < 12 or bbox[3] < 12:
            skipped += 1
            continue

        kpts_flat = ann.get('keypoints', [])
        if len(kpts_flat) < 17*3:
            skipped += 1
            continue

        img_id   = ann['image_id']
        filename = id2file.get(img_id, '')
        img_path = AP10K_IMGS / filename
        if not img_path.exists():
            skipped += 1
            continue

        kpts_17 = np.array(kpts_flat, dtype=np.float32).reshape(17, 3)
        records.append({
            'img_path': img_path,
            'bbox'    : bbox,
            'kpts_17' : kpts_17,
            'species' : species,
        })

print(f'\nAnnotazioni per specie target:')
for sp in sorted(SPECIES_TARGET):
    print(f'  {sp:15s}: {species_counter[sp]}')
print(f'\nRecord validi totali : {len(records)}')
print(f'Record skippati     : {skipped}')

## Cella 5 — Generazione dataset YOLO-pose (con fix append label)

In [ ]:
import cv2
import shutil
import random
import time

# Pulizia preventiva — evita label duplicate/sovrascritte da run precedenti
shutil.rmtree(OUT_DIR, ignore_errors=True)
print('[*] Cartella output pulita')

random.seed(42)
random.shuffle(records)

VAL_SPLIT = 0.1
n_val   = max(1, int(len(records) * VAL_SPLIT))
val_recs = records[:n_val]
trn_recs = records[n_val:]
print(f'Train: {len(trn_recs)}  Val: {len(val_recs)}')

def process_record(rec, out_dir, split):
    img = cv2.imread(str(rec['img_path']))
    if img is None:
        return False
    H, W = img.shape[:2]

    x, y, bw, bh = rec['bbox']
    cx = (x + bw/2) / W
    cy = (y + bh/2) / H
    nw = bw / W
    nh = bh / H

    kpts_20 = map_ap10k_to_fp20(rec['kpts_17'])

    kpts_norm = kpts_20.copy()
    kpts_norm[:, 0] /= W
    kpts_norm[:, 1] /= H
    kpts_norm[:, 2] = np.where(kpts_20[:, 2] == 0, 0,
                      np.where(kpts_20[:, 2] == 1, 1, 2))
    kpts_norm[:, 0] = np.clip(kpts_norm[:, 0], 0, 1)
    kpts_norm[:, 1] = np.clip(kpts_norm[:, 1], 0, 1)

    parts = [f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}']
    for kx, ky, kv in kpts_norm:
        parts.append(f'{kx:.6f} {ky:.6f} {int(kv)}')
    label_line = ' '.join(parts)

    stem = rec['img_path'].stem
    out_img_dir = out_dir / 'images' / split
    out_lbl_dir = out_dir / 'labels' / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    dst_img = out_img_dir / (stem + '.jpg')
    dst_lbl = out_lbl_dir / (stem + '.txt')

    if not dst_img.exists():
        shutil.copy(rec['img_path'], dst_img)

    # APPEND: accumula piu' animali nella stessa immagine
    with open(dst_lbl, 'a') as f:
        f.write(label_line + '\n')
    return True

t0 = time.time()
ok = err = 0
for split, recs in [('train', trn_recs), ('val', val_recs)]:
    print(f'\n[*] {split}...')
    for i, rec in enumerate(recs):
        if process_record(rec, OUT_DIR, split):
            ok += 1
        else:
            err += 1
        if i % 500 == 0 and i > 0:
            print(f'  [{i}/{len(recs)}] ok={ok} err={err}')
    print(f'  [OK] {split} completato')

print(f'\n[DONE] {ok} generati, {err} errori — {(time.time()-t0)/60:.1f}min')
for split in ['train', 'val']:
    n_img = len(list((OUT_DIR / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((OUT_DIR / 'labels' / split).glob('*.txt')))
    print(f'  {split}: {n_img} immagini, {n_lbl} file label')

## Cella 6 — Scrivi data_fase3_animals.yaml

In [ ]:
import yaml

data_yaml = {
    'path'      : str(OUT_DIR),
    'train'     : 'images/train',
    'val'       : 'images/val',
    'nc'        : 1,
    'names'     : ['animal'],
    'kpt_shape' : [N_KPT, 3],
}
yaml_path = KAGGLE_WORKING / 'data_fase3_animals.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print(f'[OK] {yaml_path}')
print(open(yaml_path).read())

## Cella 7 — Training YOLO11-Pose Animali

**Nota sulla convergenza**: test di debug con 2 epoche ha mostrato
`pose_loss` molto piu' alta di `box_loss` (9.8 vs 1.4). La pose
estimation su 20 keypoints / 8 specie eterogenee e' un task piu'
difficile della semplice detection e richiede molte piu' epoche.
Si usa `epochs=100` con `patience=30` per consentire early stopping
se converge prima.

In [ ]:
from ultralytics import YOLO
import shutil
from pathlib import Path

# ---- PARAMETRI ----
MODEL_SIZE = 'n'   # 'n'=nano (consigliato per prima verifica convergenza)
                   # passare a 's'/'l' solo dopo aver confermato che
                   # pose_loss scende in modo consistente
EPOCHS     = 100
BATCH      = 16
IMGSZ      = 640
LR0        = 0.01  # LR standard: partiamo da COCO non da SAR
LRF        = 0.01
PATIENCE   = 30
# -------------------

ckpt_dir = KAGGLE_WORKING / 'checkpoints_fase3'
ckpt_dir.mkdir(exist_ok=True)

def save_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % 10 == 0:
        src = Path(trainer.save_dir) / 'weights' / 'last.pt'
        dst = ckpt_dir / f'epoch{epoch:03d}.pt'
        shutil.copy(src, dst)
        print(f'  [CHECKPOINT] epoch{epoch:03d}.pt')

# IMPORTANTE: pesi COCO ufficiali — NON pesi Fase 2b
# L'head deve essere inizializzato nativo per 20kpt output
print(f'[*] Caricamento yolo11{MODEL_SIZE}-pose.pt (COCO ufficiale)...')
model = YOLO(f'yolo11{MODEL_SIZE}-pose.pt')
model.add_callback('on_train_epoch_end', save_checkpoint)

results = model.train(
    data         = str(yaml_path),
    epochs       = EPOCHS,
    batch        = BATCH,
    imgsz        = IMGSZ,
    lr0          = LR0,
    lrf          = LRF,
    warmup_epochs= 5,
    cos_lr       = True,
    name         = f'flypose_animal_{MODEL_SIZE}',
    project      = str(KAGGLE_WORKING / 'runs_fase3'),
    exist_ok     = True,
    device       = 0,
    workers      = 4,
    patience     = PATIENCE,
    save         = True,
    plots        = True,
    # Augmentation moderata — degrees=45 rimosso dal test di debug
    # poiche' non era la causa del problema (verificato con OKS sigma test)
    # ma teniamo comunque qualche rotazione moderata per robustezza zenitale
    degrees      = 15.0,
    fliplr       = 0.5,
    flipud       = 0.0,
    scale        = 0.5,
    mosaic       = 1.0,
)

print(f'[OK] Training completato')
print(f'Best: {results.save_dir}/weights/best.pt')

## Cella 8 — Valutazione formale

In [ ]:
from ultralytics import YOLO

run_dir = KAGGLE_WORKING / 'runs_fase3' / f'flypose_animal_{MODEL_SIZE}'
best_pt = run_dir / 'weights' / 'best.pt'
model   = YOLO(str(best_pt))
metrics = model.val(data=str(yaml_path), imgsz=IMGSZ, batch=BATCH, device=0)

print(f'\n=== Risultati Fase 3 — YOLO11{MODEL_SIZE.upper()} Animali ===')
print(f'Box  mAP@0.5  : {metrics.box.map50:.4f}')
print(f'Pose mAP@0.5  : {metrics.pose.map50:.4f}')
print(f'Precision     : {metrics.box.mp:.4f}')
print(f'Recall        : {metrics.box.mr:.4f}')
print(f'\nStream Persone (Fase 2b) per confronto:')
print(f'  Box  mAP@0.5 : 0.8541')
print(f'  Pose mAP@0.5 : 0.5790')

# Controlla l'andamento della pose_loss nel training — deve essere
# scesa significativamente sotto i valori del test di debug (9.8->8.4)
import pandas as pd
df = pd.read_csv(run_dir / 'results.csv')
print(f'\nPose loss iniziale: {df["train/pose_loss"].iloc[0]:.3f}')
print(f'Pose loss finale  : {df["train/pose_loss"].iloc[-1]:.3f}')
print(f'Pose mAP50 finale : {df["metrics/mAP50(P)"].iloc[-1]:.4f}')

## Cella 9 — Zip e download

In [ ]:
import zipfile

run_dir  = KAGGLE_WORKING / 'runs_fase3' / f'flypose_animal_{MODEL_SIZE}'
zip_path = KAGGLE_WORKING / f'flypose_animal_{MODEL_SIZE}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pt in (run_dir / 'weights').glob('*.pt'):
        zf.write(pt, f'weights/{pt.name}')
        print(f'  [+] {pt.name}')
    for pt in ckpt_dir.glob('*.pt'):
        zf.write(pt, f'checkpoints/{pt.name}')
        print(f'  [+] checkpoints/{pt.name}')
    for img in run_dir.glob('*.png'):
        zf.write(img, f'plots/{img.name}')
    csv = run_dir / 'results.csv'
    if csv.exists():
        zf.write(csv, 'results.csv')

print(f'\n[OK] {zip_path} — {zip_path.stat().st_size/1e6:.1f} MB')
print('Scarica da Output (pannello destro)')